# Phase 5 — L2 ST-GCN 模型训练与 Relay 部署
# L2 ST-GCN Training & Relay Server Deployment

**EchoGlove Edge-AI Data Glove V3**

本文档覆盖：
1. 图卷积网络（GCN）基础与手部骨架图定义
2. 时空图卷积（ST-GCN）架构详解
3. 伪骨架投影（Pseudo-Skeleton Projection）
4. ST-GCN 训练与评估
5. 置信度路由（L1 → L2 Confidence Routing）
6. Relay 服务器集成测试
7. 端到端延迟分析

---

## L1 vs L2 架构

```
ESP32-S3 (Edge)         Python Relay (Server)      Web Frontend
┌──────────────┐        ┌──────────────────┐      ┌───────────┐
│ L1 CNN+Attn  │ UDP    │ L2 ST-GCN        │ WS   │ React+R3F │
│ (34K params) │───────→│ (280K params)    │─────→│ 3D Hand   │
│ <3ms latency │ 8888   │ <20ms latency    │ 8765 │ Skeleton  │
└──────────────┘        │ NLP + TTS        │      └───────────┘
                        └──────────────────┘
```

L2 模型在以下情况触发：
- L1 置信度 ≤ 0.85
- 滑动窗口已满（30帧）
- 防抖间隔 ≥ 3 帧

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import json
import os
import time
from pathlib import Path
from typing import Tuple, Dict, Any, List, Optional

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'CLAUDE.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. 手部骨架图定义

### 1.1 图结构

将 21 个传感器通道映射为 21 个图节点，模拟 MediaPipe 手部关键点拓扑：

```
        4   8   12  16  20
        |   |   |   |   |
    3   7   11  15  19
    |   |   |   |   |
    2   6   10  14  18
    |   |   |   |   |
    1   5   9   13  17
     \   |   |   |   /
      \  |   |   |  /
         0 (wrist)
```

### 1.2 邻接矩阵

图 $G = (V, E)$ 的邻接矩阵 $\mathbf{A} \in \mathbb{R}^{N \times N}$（$N=21$），包含：
- 骨架边（父子连接）
- 掌部交叉连接（指根之间）
- 自环（每个节点到自身）

**归一化邻接矩阵**（对称归一化）：

$$
\hat{\mathbf{A}} = \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2}
$$

其中 $\mathbf{D}$ 是度矩阵：$D_{ii} = \sum_j A_{ij}$

In [ ]:
# ============================================================
# 手部骨架图定义
# 对应 glove_relay/src/models/stgcn_model.py
# ============================================================

NUM_NODES = 21

# MediaPipe 手部拓扑（21 节点）
ADJACENCY = [
    (0, 1), (1, 2), (2, 3), (3, 4),           # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),           # index
    (0, 9), (9, 10), (10, 11), (11, 12),      # middle
    (0, 13), (13, 14), (14, 15), (15, 16),    # ring
    (0, 17), (17, 18), (18, 19), (19, 20),    # pinky
    (5, 9), (9, 13), (13, 17),                 # palm cross-links
]

# 节点名称
NODE_NAMES = [
    'wrist',
    'thumb_cmc', 'thumb_mcp', 'thumb_ip', 'thumb_tip',
    'index_mcp', 'index_pip', 'index_dip', 'index_tip',
    'middle_mcp', 'middle_pip', 'middle_dip', 'middle_tip',
    'ring_mcp', 'ring_pip', 'ring_dip', 'ring_tip',
    'pinky_mcp', 'pinky_pip', 'pinky_dip', 'pinky_tip',
]

def build_adjacency_matrix(num_nodes: int = NUM_NODES) -> torch.Tensor:
    """
    构建归一化邻接矩阵 (N, N)。
    
    包含：骨架边 + 掌部交叉 + 自环
    归一化: D^{-1/2} A D^{-1/2}
    """
    adj = torch.zeros(num_nodes, num_nodes)
    for u, v in ADJACENCY:
        adj[u, v] = 1.0
        adj[v, u] = 1.0
    # 自环
    adj += torch.eye(num_nodes)
    # 对称归一化
    deg = adj.sum(dim=-1, keepdim=True).clamp(min=1)
    deg_inv_sqrt = torch.pow(deg, -0.5)
    adj_norm = deg_inv_sqrt * adj * deg_inv_sqrt.T
    return adj_norm


A_norm = build_adjacency_matrix()
print(f'Adjacency matrix shape: {A_norm.shape}')
print(f'Non-zero entries: {(A_norm > 0).sum().item()}')
print(f'Max value: {A_norm.max():.4f}')
print(f'Min non-zero: {A_norm[A_norm > 0].min():.4f}')

In [ ]:
# ---- 骨架图可视化 ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 1. 骨架拓扑图
# 简化的 2D 手部布局
positions = {
    0: (0, 0),   # wrist
    1: (-0.3, 0.5), 2: (-0.4, 1.0), 3: (-0.5, 1.5), 4: (-0.6, 2.0),  # thumb
    5: (-0.15, 0.6), 6: (-0.15, 1.2), 7: (-0.15, 1.8), 8: (-0.15, 2.3),  # index
    9: (0, 0.6), 10: (0, 1.2), 11: (0, 1.8), 12: (0, 2.3),  # middle
    13: (0.15, 0.6), 14: (0.15, 1.2), 15: (0.15, 1.8), 16: (0.15, 2.3),  # ring
    17: (0.3, 0.5), 18: (0.35, 1.0), 19: (0.4, 1.5), 20: (0.45, 2.0),  # pinky
}

# 绘制边
for u, v in ADJACENCY:
    x0, y0 = positions[u]
    x1, y1 = positions[v]
    color = '#e74c3c' if u == 0 or v == 0 else '#3498db'  # 腕部连接红色
    ax1.plot([x0, x1], [y0, y1], color=color, linewidth=1.5, alpha=0.6)

# 绘制节点
for i, (x, y) in positions.items():
    color = '#e74c3c' if i == 0 else '#3498db'
    ax1.scatter(x, y, s=100, color=color, zorder=5)
    ax1.annotate(str(i), (x, y), textcoords='offset points',
                xytext=(5, 5), fontsize=7)

ax1.set_title('Hand Skeleton Graph (21 Nodes)')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.2)

# 2. 邻接矩阵热力图
im = ax2.imshow(A_norm.numpy(), cmap='Blues', vmin=0)
ax2.set_title('Normalized Adjacency Matrix $\\hat{A}$')
ax2.set_xlabel('Node j')
ax2.set_ylabel('Node i')
plt.colorbar(im, ax=ax2, fraction=0.046)

plt.tight_layout()
plt.show()

## 2. 图卷积层（Graph Convolution）

### 2.1 谱域图卷积

图卷积的谱域定义：

$$
\mathbf{y} = \mathbf{U} g_\theta(\mathbf{\Lambda}) \mathbf{U}^T \mathbf{x}
$$

其中 $\mathbf{U}$ 是归一化拉普拉斯矩阵的特征向量，$\mathbf{\Lambda}$ 是特征值对角矩阵。

### 2.2 一阶近似（Chebyshev → GCN）

简化为一阶切比雪夫多项式近似：

$$
\mathbf{y} = \hat{\mathbf{A}} \mathbf{x} \mathbf{W} + \mathbf{b}
$$

其中：
- $\hat{\mathbf{A}} \in \mathbb{R}^{N \times N}$: 归一化邻接矩阵
- $\mathbf{x} \in \mathbb{R}^{N \times C_{in}}$: 节点特征
- $\mathbf{W} \in \mathbb{R}^{C_{in} \times C_{out}}$: 可学习权重
- $\mathbf{b} \in \mathbb{R}^{C_{out}}$: 偏置

**计算复杂度**: $O(N^2 C_{in} + N C_{in} C_{out})$

In [ ]:
class GraphConv(nn.Module):
    """
    空间图卷积层。
    
    forward: y = A @ x @ W + b
    """
    
    def __init__(self, in_channels: int, out_channels: int, num_nodes: int = NUM_NODES):
        super().__init__()
        self.num_nodes = num_nodes
        # 可学习邻接矩阵（初始化为归一化拓扑）
        self.A = nn.Parameter(build_adjacency_matrix(num_nodes))
        self.weight = nn.Parameter(torch.randn(in_channels, out_channels) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_channels))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, N, C_in)
        returns: (B, N, C_out)
        """
        # support = A @ x: (N,N) @ (B,N,C_in) = (B,N,C_in)
        support = torch.einsum('nm,bmc->bnc', self.A, x)
        # out = support @ W + b
        return support @ self.weight + self.bias


# 验证
gconv = GraphConv(2, 64)
x_test = torch.randn(4, NUM_NODES, 2)
y_test = gconv(x_test)
print(f'GraphConv: {x_test.shape} → {y_test.shape}')

## 3. 时空图卷积块（ST-Conv Block）

### 3.1 结构

```
Input (B, T, N, C_in)
    │
    ├──── Spatial ──────────────────────────┐
    │   Reshape to (B*T, N, C_in)           │
    │   GraphConv → BN → ReLU               │
    │   Reshape to (B, T, N, C_out)         │
    │                                        │
    ├──── Temporal ─────────────────────────┤
    │   Reshape to (B*N, C_out, T)          │
    │   Conv1d → BN → ReLU                  │
    │   Reshape to (B, T, N, C_out)         │
    │                                        │
    └──── Residual ─────────────────────────┘
        if C_in != C_out:
            identity = Conv1d_1x1(identity)
        output = ReLU(temporal + identity)
```

### 3.2 时序卷积

使用膨胀卷积扩大时序感受野：

$$
y_t = \text{ReLU}(\text{BN}(\sum_{k=0}^{K-1} w_k \cdot x_{t - k \cdot d}))
$$

| ST-Conv Block | Dilation | 感受野 |
|---------------|----------|--------|
| Block 1 | 1 | 3 frames (30ms) |
| Block 2 | 2 | 5 frames (50ms) |
| Block 3 | 1 | 3 frames (30ms) |

In [ ]:
class TemporalConv(nn.Module):
    """时序卷积层（TCN 风格）。"""
    
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              padding=padding, dilation=dilation)
        self.bn = nn.BatchNorm1d(out_channels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, T) → (B, C_out, T)"""
        return F.relu(self.bn(self.conv(x)))


class STConvBlock(nn.Module):
    """时空图卷积块：Spatial → BN → ReLU → Temporal → BN → ReLU + Residual"""
    
    def __init__(self, in_channels: int, out_channels: int,
                 num_nodes: int = NUM_NODES,
                 temporal_kernel: int = 3,
                 temporal_dilation: int = 1):
        super().__init__()
        
        self.graph_conv = GraphConv(in_channels, out_channels, num_nodes)
        self.bn_s = nn.BatchNorm1d(out_channels)
        self.temp_conv = TemporalConv(out_channels, out_channels,
                                      temporal_kernel, dilation=temporal_dilation)
        
        # 残差投影
        self.residual = None
        if in_channels != out_channels:
            self.residual = nn.Conv1d(in_channels, out_channels, 1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, T, N, C_in)
        returns: (B, T, N, C_out)
        """
        B, T, N, C = x.shape
        identity = x
        
        # ---- Spatial: per-timestep graph conv ----
        x_spatial = x.reshape(B * T, N, C)
        x_spatial = self.graph_conv(x_spatial)     # (B*T, N, C_out)
        C_out = x_spatial.shape[-1]
        x_spatial = F.relu(self.bn_s(x_spatial.permute(0, 2, 1)).permute(0, 2, 1))
        x_spatial = x_spatial.reshape(B, T, N, C_out)
        
        # ---- Temporal: per-node 1D conv ----
        x_temporal = x_spatial.permute(0, 2, 3, 1).reshape(B * N, C_out, T)
        x_temporal = self.temp_conv(x_temporal)     # (B*N, C_out, T)
        out = x_temporal.reshape(B, N, C_out, T).permute(0, 3, 1, 2)
        
        # ---- Residual ----
        if self.residual is not None:
            identity = identity.permute(0, 3, 2, 1).reshape(B * N, C, T)
            identity = self.residual(identity)
            identity = identity.reshape(B, N, C_out, T).permute(0, 3, 1, 2)
        
        return F.relu(out + identity)


# 验证
stblock = STConvBlock(2, 64)
x_test = torch.randn(4, 30, NUM_NODES, 2)
y_test = stblock(x_test)
print(f'STConvBlock: {x_test.shape} → {y_test.shape}')

## 4. ST-GCN 完整模型

### 4.1 架构流程

```
Input (B, 30, 21)                         ← 滑动窗口
    │
    ▼ Linear(21 → 42) + LayerNorm         ← 伪骨架投影
(B, 30, 42)
    │
    ▼ Reshape
(B, 30, 21, 2)                             ← 21节点×2D坐标
    │
    ▼ ST-Conv Block 1 (2→64, dilation=1)
(B, 30, 21, 64)
    │
    ▼ ST-Conv Block 2 (64→64, dilation=2)
(B, 30, 21, 64)
    │
    ▼ ST-Conv Block 3 (64→128, dilation=1)
(B, 30, 21, 128)
    │
    ▼ AttentionPooling (B, 128)            ← 时空注意力池化
    │
    ▼ Dropout(0.4) → Linear(128 → 46)
(B, 46) logits
```

### 4.2 伪骨架投影

将 21 维传感器特征映射为 21 个伪 2D 关键点坐标：

$$
\mathbf{P} = \text{LayerNorm}(\mathbf{x} \cdot \mathbf{W}_{\text{proj}}) \in \mathbb{R}^{21 \times 2}
$$

这使得传感器数据可以被解释为手部关节的空间坐标，从而利用图卷积的空间推理能力。

### 4.3 注意力池化

$$
\alpha_{t,n} = \text{softmax}(\mathbf{W}_2 \tanh(\mathbf{W}_1 \mathbf{h}_{t,n}))
$$

$$
\mathbf{z} = \sum_{t,n} \alpha_{t,n} \cdot \mathbf{h}_{t,n}
$$

将 `(B, T, N, C)` 时空特征聚合为 `(B, C)` 分类向量。

In [ ]:
class AttentionPooling(nn.Module):
    """时空注意力池化: (B, T, N, C) → (B, C)"""
    
    def __init__(self, in_channels: int):
        super().__init__()
        self.att = nn.Sequential(
            nn.Linear(in_channels, in_channels // 4),
            nn.Tanh(),
            nn.Linear(in_channels // 4, 1),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, N, C) → (B, C)"""
        B, T, N, C = x.shape
        x_flat = x.reshape(B * T * N, C)
        weights = self.att(x_flat).reshape(B, T, N)
        weights = torch.softmax(weights.reshape(B, -1), dim=-1).reshape(B, T, N, 1)
        return (x * weights).sum(dim=(1, 2))


class STGCNModel(nn.Module):
    """
    时空图卷积网络 (≈280K params)。
    输入: (B, T, 21) 滑动窗口
    输出: (B, num_classes) logits
    """
    
    def __init__(self, input_dim: int = 21, hidden_dim: int = 64,
                 num_classes: int = 46, num_frames: int = 30,
                 num_nodes: int = NUM_NODES):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_frames = num_frames
        self.num_nodes = num_nodes
        
        # 伪骨架投影: 21 → 21×2 = 42
        self.skeleton_proj = nn.Linear(input_dim, num_nodes * 2)
        self.coord_norm = nn.LayerNorm(num_nodes * 2)
        
        # ST-Conv 块
        self.st_blocks = nn.ModuleList([
            STConvBlock(2, hidden_dim, num_nodes, temporal_dilation=1),
            STConvBlock(hidden_dim, hidden_dim, num_nodes, temporal_dilation=2),
            STConvBlock(hidden_dim, hidden_dim * 2, num_nodes, temporal_dilation=1),
        ])
        
        # 注意力池化 + 分类头
        self.attn_pool = AttentionPooling(hidden_dim * 2)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, T, 21)
        returns: (B, num_classes)
        """
        B, T, _ = x.shape
        
        # 伪骨架投影
        x = self.skeleton_proj(x)               # (B, T, 42)
        x = self.coord_norm(x)
        x = x.reshape(B, T, self.num_nodes, 2)  # (B, T, 21, 2)
        
        # ST-Conv 块
        for block in self.st_blocks:
            x = block(x)
        
        # 注意力池化
        x = self.attn_pool(x)  # (B, C)
        
        # 分类
        x = self.dropout(x)
        return self.fc(x)
    
    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


# 实例化
NUM_CLASSES = 20  # 合成数据集
model = STGCNModel(input_dim=21, hidden_dim=64, num_classes=NUM_CLASSES)
print(f'ST-GCN parameters: {model.count_params():,}')

dummy_in = torch.randn(4, 30, 21)
dummy_out = model(dummy_in)
print(f'Input: {dummy_in.shape} → Output: {dummy_out.shape}')

In [ ]:
# ---- 层级形状追踪 ----
model.eval()
x = torch.randn(1, 30, 21)

print('ST-GCN Layer-by-Layer Shape Trace:')
print('-' * 55)

B, T, _ = x.shape
print(f'Input:              {x.shape}')

h = model.skeleton_proj(x)
print(f'Skeleton projection: {h.shape}')

h = model.coord_norm(h)
h = h.reshape(B, T, NUM_NODES, 2)
print(f'Coord reshape:      {h.shape}')

for i, block in enumerate(model.st_blocks):
    h = block(h)
    print(f'ST-Conv Block {i+1}:    {h.shape}')

h = model.attn_pool(h)
print(f'Attention pooling:  {h.shape}')

h = model.dropout(h)
h = model.fc(h)
print(f'FC output:          {h.shape}')

## 5. ST-GCN 训练

### 5.1 超参数

| 参数 | 值 | 说明 |
|------|-----|------|
| Optimizer | AdamW | lr=1e-3, weight_decay=1e-4 |
| Scheduler | CosineAnnealing | $T_{\max}$ = epochs |
| Dropout | 0.4 | 防止过拟合 |
| Batch size | 64 | |
| Epochs | 80 | Early stopping patience=15 |

In [ ]:
# ---- 加载数据 ----
data_path = PROJECT_ROOT / 'data' / 'processed' / 'synthetic_dataset.npz'
if data_path.exists():
    data = np.load(data_path, allow_pickle=True)
    X_train = data['X_train']
    y_train = data['y_train']
    X_val = data['X_val']
    y_val = data['y_val']
    NUM_CLASSES = int(data['num_classes'])
else:
    # 内联生成
    np.random.seed(42)
    N = 2000
    X_train = np.random.randn(N, 30, 21).astype(np.float32) * 0.3 + 0.5
    y_train = np.random.randint(0, NUM_CLASSES, N)
    X_val = np.random.randn(500, 30, 21).astype(np.float32) * 0.3 + 0.5
    y_val = np.random.randint(0, NUM_CLASSES, 500)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
val_ds = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

print(f'Train: {len(train_ds)} samples, Val: {len(val_ds)} samples')
print(f'Classes: {NUM_CLASSES}')

In [ ]:
# ---- 训练 ----
model = STGCNModel(input_dim=21, hidden_dim=64, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0

print('Training ST-GCN...')
for epoch in range(1, 81):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(-1) == yb).sum().item()
        train_total += xb.size(0)
    scheduler.step()
    
    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(-1) == yb).sum().item()
            val_total += xb.size(0)
    
    tl = train_loss / train_total
    ta = train_correct / train_total
    vl = val_loss / val_total
    va = val_correct / val_total
    
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    
    if va > best_val_acc:
        best_val_acc = va
        patience_counter = 0
        save_dir = PROJECT_ROOT / 'checkpoints'
        save_dir.mkdir(exist_ok=True)
        torch.save(model.state_dict(), save_dir / 'l2_stgcn_best.pt')
        marker = ' ★'
    else:
        patience_counter += 1
        marker = ''
    
    if epoch % 10 == 0 or marker:
        print(f'  Epoch {epoch:3d}/80 | Train: {tl:.4f}/{ta:.4f} | Val: {vl:.4f}/{va:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}{marker}')
    
    if patience_counter >= 15:
        print(f'  Early stopping at epoch {epoch}')
        break

print(f'\nBest val accuracy: {best_val_acc:.4f}')

In [ ]:
# ---- 训练曲线 ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(history['train_loss'], label='Train', color='#e74c3c')
ax1.plot(history['val_loss'], label='Val', color='#3498db')
ax1.set_title('ST-GCN Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', color='#e74c3c')
ax2.plot(history['val_acc'], label='Val', color='#3498db')
ax2.set_title('ST-GCN Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 置信度路由（Confidence Routing）

### 6.1 路由策略

L1 和 L2 之间的切换由**置信度路由器**控制：

$$
\text{decision} = \begin{cases}
\text{L1 output} & \text{if } p_{\text{L1}}(\hat{y}) > \tau_{\text{L1}} = 0.85 \\
\text{L2 output} & \text{otherwise}
\end{cases}
$$

### 6.2 防抖机制

- **L2 触发防抖**: 连续 3 帧 L1 低置信度才触发 L2
- **手势间隔**: 800ms 静默期分隔不同手势
- **L2 滑动窗口**: 30 帧缓冲

### 6.3 配置参数

```yaml
# relay_config.yaml
l1_confidence_threshold: 0.85
l2_debounce_frames: 3
gesture_silence_ms: 800
l2_window_size: 30
```

In [ ]:
class ConfidenceRouter:
    """
    L1/L2 置信度路由器。
    
    Parameters
    ----------
    l1_threshold : float
        L1 置信度阈值，低于此值触发 L2
    debounce_frames : int
        L2 触发防抖帧数
    silence_ms : int
        手势间隔（ms）
    """
    
    def __init__(self, l1_threshold: float = 0.85,
                 debounce_frames: int = 3,
                 silence_ms: int = 800):
        self.l1_threshold = l1_threshold
        self.debounce_frames = debounce_frames
        self.silence_ms = silence_ms
        
        self._low_conf_count = 0
        self._last_gesture_time = 0
        self._l2_buffer = []
    
    def route(self, l1_result: Tuple[int, float],
              timestamp_ms: float) -> Dict:
        """
        路由决策。
        
        Parameters
        ----------
        l1_result : (gesture_id, confidence)
            L1 模型的预测结果
        timestamp_ms : float
            当前时间戳
            
        Returns
        -------
        dict
            {'source': 'L1'|'L2', 'gesture_id': int, 'confidence': float}
        """
        gesture_id, confidence = l1_result
        
        # 手势间隔检查
        if timestamp_ms - self._last_gesture_time < self.silence_ms:
            return {'source': 'silence', 'gesture_id': -1, 'confidence': 0}
        
        # L1 高置信度：直接输出
        if confidence > self.l1_threshold:
            self._low_conf_count = 0
            self._last_gesture_time = timestamp_ms
            return {'source': 'L1', 'gesture_id': gesture_id, 'confidence': confidence}
        
        # L1 低置信度：累计计数
        self._low_conf_count += 1
        
        if self._low_conf_count >= self.debounce_frames:
            # 触发 L2
            self._low_conf_count = 0
            self._last_gesture_time = timestamp_ms
            return {'source': 'L2', 'gesture_id': -1, 'confidence': 0}  # L2 待处理
        
        return {'source': 'waiting', 'gesture_id': -1, 'confidence': confidence}


# ---- 模拟路由场景 ----
router = ConfidenceRouter()

# 模拟 L1 输出序列
scenarios = [
    (10, 0.95),   # L1 高置信
    (10, 0.92),   # L1 高置信
    (15, 0.60),   # L1 低置信 (开始)
    (15, 0.55),   # L1 低置信 (计数+1)
    (15, 0.70),   # L1 低置信 (计数+2 → 触发L2)
    (15, 0.50),   # L1 低置信 (L2已触发)
]

print('Confidence Routing Simulation:')
print('-' * 70)
print(f'{"L1 Result":>15} | {"Router Decision":>20} | {"Low Conf Count":>14}')
print('-' * 70)

for i, (gid, conf) in enumerate(scenarios):
    result = router.route((gid, conf), i * 100)
    print(f'  ({gid:2d}, {conf:.2f})  | {result["source"]:>18} | {router._low_conf_count:>12}')

## 7. Relay 服务器集成测试

### 7.1 数据流

```
ESP32 UDP:8888 → Protobuf → JSON → L1 Inference
                                     │
                                     ▼
                              ConfidenceRouter
                                     │
                              ┌──────┴──────┐
                              │             │
                           L1 high       L1 low
                              │             │
                              ▼             ▼
                          Gesture ID    L2 ST-GCN
                              │             │
                              ▼             ▼
                           NLP → TTS → WebSocket:8765
```

### 7.2 模拟端到端推理

In [ ]:
# ---- 端到端推理模拟 ----
model.eval()

# 加载手势标签
labels_path = PROJECT_ROOT / 'glove_relay' / 'data' / 'gesture_labels.json'
with open(labels_path) as f:
    gesture_labels = json.load(f)

# 模拟从传感器到推理的完整流程
def simulate_relay_inference(model, num_steps=5):
    """模拟 Relay 端推理流程。"""
    model.eval()
    results = []
    
    for step in range(num_steps):
        # 模拟传感器数据
        raw_sensor = torch.randn(1, 30, 21) * 0.3 + 0.5
        
        # L2 ST-GCN 推理
        with torch.no_grad():
            start = time.perf_counter()
            logits = model(raw_sensor)
            latency_ms = (time.perf_counter() - start) * 1000
            
            probs = F.softmax(logits, dim=-1)
            confidence, pred_id = probs.max(dim=-1)
        
        pred_id = pred_id.item()
        confidence = confidence.item()
        
        # 查找标签
        label = next((g for g in gesture_labels if g['id'] == pred_id), None)
        label_str = f'{label["name_cn"]} ({label["name_en"]})' if label else f'Class {pred_id}'
        
        results.append({
            'step': step,
            'prediction': pred_id,
            'label': label_str,
            'confidence': confidence,
            'latency_ms': latency_ms,
        })
        
        print(f'Step {step}: Pred={pred_id:2d} ({label_str:>20s}) | '
              f'Conf={confidence:.3f} | Latency={latency_ms:.2f}ms')
    
    return results


print('Relay Inference Simulation:')
print('=' * 70)
results = simulate_relay_inference(model, num_steps=5)
print()
print(f'Average latency: {np.mean([r["latency_ms"] for r in results]):.2f} ms')

## 8. 端到端延迟分析

### 8.1 延迟预算

| 阶段 | 目标 | 说明 |
|------|------|------|
| 传感器采样 | 10 ms | 100 Hz |
| 卡尔曼滤波 | < 0.1 ms | 21 通道 |
| L1 推理 | < 3 ms | ESP32-S3 TFLite |
| UDP 传输 | < 1 ms | 局域网 |
| L2 推理 | < 20 ms | Python PyTorch |
| NLP + TTS | < 50 ms | 服务端 |
| WebSocket | < 5 ms | 局域网 |
| **总计** | **< 100 ms** | **端到端** |

### 8.2 延迟分布

In [ ]:
# ---- 延迟基准测试 ----
def benchmark_model(model, input_shape, num_runs=500, warmup=50):
    """基准测试模型推理延迟。"""
    model.eval().to(DEVICE)
    dummy_input = torch.randn(1, *input_shape).to(DEVICE)
    
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy_input)
    
    # Benchmark
    latencies = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(dummy_input)
            latencies.append((time.perf_counter() - start) * 1000)
    
    return np.array(latencies)


# 测试 L2 ST-GCN
latencies = benchmark_model(model, (30, 21))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 直方图
ax1.hist(latencies, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
ax1.axvline(x=np.median(latencies), color='red', linestyle='--', 
            label=f'Median: {np.median(latencies):.2f} ms')
ax1.axvline(x=np.percentile(latencies, 95), color='orange', linestyle='--',
            label=f'P95: {np.percentile(latencies, 95):.2f} ms')
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('Count')
ax1.set_title('L2 ST-GCN Inference Latency Distribution')
ax1.legend()

# 端到端延迟堆叠条
stages = ['Sensor\n(10ms)', 'Kalman\n(0.1ms)', 'L1\n(3ms)', 'UDP\n(1ms)',
          'L2\n(~{:.1f}ms)'.format(np.median(latencies)),
          'NLP+TTS\n(50ms)', 'WS\n(5ms)']
delays = [10, 0.1, 3, 1, np.median(latencies), 50, 5]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']

ax2.barh(range(len(stages)), delays, color=colors, alpha=0.8)
ax2.set_yticks(range(len(stages)))
ax2.set_yticklabels(stages)
ax2.set_xlabel('Latency (ms)')
ax2.set_title('End-to-End Latency Breakdown')
ax2.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='100ms target')
ax2.legend()

plt.tight_layout()
plt.show()

total = sum(delays)
print(f'\nEnd-to-end latency estimate: {total:.1f} ms')
print(f'Target: 100 ms → {"PASS ✓" if total < 100 else "NEEDS OPTIMIZATION ✗"}')

## 9. 三模型对比总结

| 指标 | L1 CNN+Attn | L1 MS-TCN | L2 ST-GCN |
|------|-------------|-----------|------------|
| 参数量 | ~34K | ~12K | ~280K |
| 输入 | (B, 21) | (B, T, 21) | (B, 30, 21) |
| 推理延迟 | < 3 ms | < 1 ms | < 20 ms |
| 目标精度 | > 90% | > 90% | > 95% |
| 部署位置 | ESP32-S3 | ESP32-S3 | Python Relay |
| 量化格式 | TFLite INT8 | TFLite INT8 | PyTorch FP32 |
| 模型大小 | ~34 KB | ~12 KB | ~1.1 MB |

---

## 小结

| 组件 | 算法 | 关键公式 |
|------|------|----------|
| 图卷积 | GCN | $\mathbf{y} = \hat{\mathbf{A}} \mathbf{x} \mathbf{W} + \mathbf{b}$ |
| 时空块 | ST-Conv | Spatial GCN → Temporal Conv + Residual |
| 伪骨架 | Linear proj | $\mathbb{R}^{21} \to \mathbb{R}^{21 \times 2}$ |
| 注意力池化 | Soft attention | $\alpha = \text{softmax}(\mathbf{W}_2 \tanh(\mathbf{W}_1 h))$ |
| 置信度路由 | Threshold + debounce | $\tau = 0.85$, debounce = 3 frames |

**所有三个 Notebook 的数据流**：

```
01_data_collection → 02_l1_training → 03_l2_stgcn → ESP32部署 + Relay部署
```